<a href="https://colab.research.google.com/github/dayana-rb/Ingesti-n_de_Datos_desde_un_API/blob/main/EA1_Ingesti%C3%B3n_de_Datos_desde_un_API_(2)_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**<span style="color:#809bd8">Curso:</span>** **Introducción a Ingeniería de Sistemas**

**<span style="color:#809bd8">Tema:</span>** **Evidencia de Aprendizaje EA1**

**<span style="color:#809bd8">Estudiante:</span>** **Deisy Dayana Bueno Fernandez**

**<span style="color:#809bd8">Profesor:</span>** **Walter Hugo Arboleda Mazo**

**<span style="color:#809bd8">Universidad:</span>** **UNAC**

**<span style="color:#809bd8">Fecha:</span>** **04/03/2026**

In [ ]:
# pip install requests pandas openpyxl

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import requests # importacion de la libreria Requests para enviar peticion a la API https://jsonplaceholder.typicode.com/users
#API significa “interfaz de programación de aplicaciones”.
#Las API son mecanismos que permiten a dos componentes de software comunicarse entre sí mediante un conjunto de definiciones y protocolos.
import sqlite3 # importacion de la interface para conexion y creacion de bases de datos SQLite
#SQLite es una base de datos relacional que se ejecuta en la misma instancia que la aplicación, sin necesidad de un servidor.
import pandas as pd # importacion de Pandas, libreria para el analisis de datos en Python
# pandas es un paquete de manipulación de datos en Python para datos tabulares.
# Es decir, datos en forma de filas y columnas, también conocidos como DataFrames

# Definicion de la URL de la API para envio de peticion y recepcion de datos usando protocolo HTTP
# desde la API https://jsonplaceholder.typicode.com/
#URL significa Uniform Resource Locator (Localizador de Recursos Uniforme).Es el mecanismo usado por los navegadores para obtener cualquier recurso publicado en la web.
API_URL = "https://jsonplaceholder.typicode.com/users" #Guarda en la variable API_URL la dirección de la API desde donde se van a extraer los datos

# Creacion de la funcion extraer_datos validandose codigo 200 (exito en la conexion) o codigo 404 (codigo de error en la conexion)
def extraer_datos(url):#Define una función llamada extraer_datos que recibe una URL como parámetro.
    response = requests.get(url)#Envía una petición GET a la URL indicada y Guarda la respuesta del servidor en la variable "response".
    # Una petición GET solicita al servidor una información o recurso concreto.
    if response.status_code == 200:#Verifica si el código de estado HTTP es 200 (conexión exitosa).
        return response.json()#Si la conexión fue exitosa, convierte la respuesta en formato JSON y la devuelve como resultado de la función.
        #JSON (JavaScript Object Notation) es un formato de texto estándar para almacenar y transferir datos estructurados
    else:#Si el código NO es 200 (hubo error)...
        raise Exception(f"Error al conectar con la API: {response.status_code}")#Lanza una excepción mostrando el código de error.


# Creacion de la base de datos en SQLite "datos_api.db"
# se realiza la conexion a dicha base de datos
# y se crea el "cursor" para "execute" el CREATE TABLE y el INSERT OR REPLACE INTO usuarios
def guardar_en_db(datos):#Define una función llamada guardar_en_db que recibe los datos obtenidos de la API.
    conn = sqlite3.connect('datos_api.db')#Crea (o abre si ya existe) una base de datos llamada datos_api.db.
    cursor = conn.cursor()#Crea un cursor que permite ejecutar comandos SQL dentro de la base de datos.

    # Creación de la tabla "usuarios" dentro de la base de datos "datos_api.db"
    # esta contiene los campos id, nombre, usuario y email
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS usuarios (
            id INTEGER PRIMARY KEY,
            nombre TEXT,
            usuario TEXT,
            email TEXT
        )
    ''')#Ejecuta un comando SQL para crear la tabla "usuarios" si no existe.
    #SQL, o Structured Query Language, es esencial para interactuar con bases de datos y realizar diversas operaciones sobre los datos.
    #La tabla tiene 4 columnas:
    #- id (clave primaria)
    #- nombre
    #- usuario
    #- email

    # Inserción de datos en la tabla "usuarios"
    for item in datos:#Recorre cada registro recibido desde la API.
        cursor.execute('''
            INSERT OR REPLACE INTO usuarios (id, nombre, usuario, email)
            VALUES (?, ?, ?, ?)
        ''', (item['id'], item['name'], item['username'], item['email']))#Inserta los datos en la tabla usuarios.
        #Si ya existe un registro con el mismo id, lo reemplaza.Los signos ? son parámetros que se reemplazan por los valores del diccionario item.

    # Aceptación mediante "Commit" el ingreso de datos en la tabla "usuarios"
    conn.commit()#Guarda definitivamente los cambios en la base de datos.

    # Cerrado de la conexion a la base de datos "datos_api.db"
    conn.close()#Cierra la conexión con la base de datos.

# 3. GENERAR EVIDENCIAS (Pandas y Auditoría)
def generar_evidencias(datos_api):#Define la función que generará los archivos de evidencia (CSV y auditoría).
    # --- Archivo Excel/CSV con Pandas ---
    conn = sqlite3.connect('datos_api.db')#Abre la base de datos nuevamente.
    df_db = pd.read_sql_query("SELECT * FROM usuarios", conn)#Lee todos los registros de la tabla usuarios y los guarda en un DataFrame de pandas.

    df_db.to_csv('muestra_usuarios.csv', index=False)#Guarda el DataFrame en un archivo CSV llamado muestra_usuarios.csv.index=False evita que se guarde la columna de índices.

    conn.close()#Cierra la conexión con la base de datos.

    # la variable "registros_api" obtiene la cantidad de registros recibidos desde la
    # API con url "https://jsonplaceholder.typicode.com/users"
    # la variable "registros_db" obtiene la cantidad de registros almacenados en la tabla "usuarios"
    registros_api = len(datos_api)#Cuenta cuántos registros se obtuvieron desde la API.
    registros_db = len(df_db)#Cuenta cuántos registros hay en la base de datos.

    # Creación del archivo "auditoría.txt" con el metodo "open" de Python
    with open('auditoria.txt', 'w', encoding='utf-8') as f:#Crea (o sobrescribe) el archivo auditoria.txt.Se abre en modo escritura ('w').

        f.write("RESUMEN DE AUDITORÍA\n")#Escribe el encabezado del archivo.
        f.write("====================\n")#Escribe el encabezado del archivo.
        f.write(f"Registros extraídos de la API https://jsonplaceholder.typicode.com/: {registros_api}\n")#Escribe la cantidad de registros obtenidos desde la API.
        f.write(f"Registros guardados en la DB de SQLite3: {registros_db}\n")#Escribe la cantidad de registros guardados en la base de datos.

        if registros_api == registros_db:#Compara si ambas cantidades son iguales.
            f.write("\nESTADO: Operacion Exitosa.\n")#Si son iguales, escribe que la operación fue exitosa.
        else:
            f.write("\nESTADO: Operacion Fallida.\n")#Si no coinciden, escribe que la operación falló.

# EJECUCIÓN DEL PROCESO
try:#Inicia un bloque try para manejar posibles errores.
    print("Iniciando proceso de petición y recepción de datos desde la API https://jsonplaceholder.typicode.com/users...")#Muestra un mensaje en pantalla indicando que inicia el proceso.
    datos = extraer_datos(API_URL)#Llama a la función extraer_datos y guarda el resultado en "datos".

    print("Creando la base de datos datos_api.db y guardando datos en la tabla usuarios...")
    guardar_en_db(datos)#Llama a la función extraer_datos y guarda el resultado en "datos".

    print("Generando archivos de evidencia .CSV y auditoría.txt...")
    generar_evidencias(datos)#Genera el archivo CSV y el archivo de auditoría.

    print("¡Proceso completado con éxito!")#Muestra mensaje de éxito.
except Exception as e:#Si ocurre cualquier error durante el proceso...
    print(f"Ocurrió un error: {e}")#Muestra el mensaje de error.

Iniciando proceso de petición y recepción de datos desde la API https://jsonplaceholder.typicode.com/users...
Creando la base de datos datos_api.db y guardando datos en la tabla usuarios...
Generando archivos de evidencia .CSV y auditoría.txt...
¡Proceso completado con éxito!



**Referencias**

https://realpython.com/python-requests/#make-a-get-request

https://pandas.pydata.org/

https://pypi.org/project/openpyxl/

https://docs.python.org/3/library/sqlite3.html

https://sqliteviewer.app/

https://sibabalwesinyaniso.medium.com/what-is-a-cursor-understanding-how-sqlite-handles-queries-in-python-8f8c88546820
https://www.pythonlore.com/understanding-sqlite3-cursor-object-for-database-operations/
https://miro.medium.com/v2/resize:fit:720/format:webp/0*rePKZ6jMZbZ_6S_3.gif
